# Four Rooms (MiniGrid) — a 7,675-parameter recurrent Q-learning agent

Solution of **`MiniGrid-FourRooms-v0`** with **Q-learning + a neural network** under a hard constraint:
**fewer than 10,000 trainable parameters**, with memory, on a fixed-goal variant of the task.

v2 of this notebook used 1,011,651 parameters. This one uses **7,675** — 0.8% as many — and the reduction
comes from one architectural change, not from shrinking widths.

---

## 1. The task variant

```python
gym.make("MiniGrid-FourRooms-v0", agent_pos=(1, 1), goal_pos=(17, 17))
```

Those kwargs fix the agent's start cell and the goal cell. **They do not fix the maze.** Reading
`FourRoomsEnv._gen_grid`, the four doorway positions are drawn with `self._rand_int(...)` on *every* reset
regardless of the kwargs, and so is the agent's initial heading:

| Quantity | Fixed by kwargs? | Variation |
|---|---|---|
| agent start cell | yes | 1 |
| goal cell | yes | 1 |
| doorway positions | **no** | 8 choices x 4 doors = **4,096 layouts** |
| agent start heading | **no** | 4 |

So this is deliberately *not* a deterministic task — 16,384 reset variants remain. A lookup table is
therefore still out of reach and function approximation is still required, which is the point: the network
has to generalise across mazes it has never seen.

> If you ever want a *genuinely* fixed layout, the kwargs are not enough — you must subclass
> `FourRoomsEnv` and override `_gen_grid` to place the doorways deterministically. That variant is much
> easier and is solvable with tabular Q-learning over hashed observations in roughly 2,600 table entries.

## 2. Why the task is much easier than the default, and why memory still earns its place

With the start cell fixed and `obs["direction"]` giving **absolute** heading for free, the agent can
**dead-reckon**: integrating its own forward moves from a known origin yields its absolute position. The
goal never moves, so "which way is the goal" is a constant the network can bake into its weights.

That reduces the problem to: *track where I am, and find the local doorway.* Compare with the default
environment, where the goal is re-placed randomly and dead reckoning buys nothing because the target is
unknown. Memory is **more** useful here than in v2, not less — it is the mechanism that converts a history
of actions into a position estimate.

Note the arc this completes. In v1 the direction one-hot was near-redundant (MiniGrid already rotates the
view into the agent's frame). In v2 it became mildly useful. Here it is **load-bearing**: dead reckoning is
impossible without absolute heading.

## 3. Where the parameters went

v2's cost was concentrated in one layer:

| v2 component | Parameters | Share |
|---|---|---|
| `Linear(1576 -> 256)` after `Flatten` | 403,712 | 39.9% |
| `LSTM(256, 256)` | 526,336 | 52.0% |
| everything else | 81,603 | 8.1% |

Flattening a 32-channel 7x7 map produces 1,568 features, and the dense layer that consumes them is 40% of
the network. **Replacing `Flatten` with a parameter-free spatial readout is the whole trick.**

`SpatialReadout` below collapses `(batch, C, 7, 7)` to `3C + 4C = 7C` numbers using no parameters at all:

* **spatial softmax** — per channel, a softmax over the 49 positions, then the expected `(x, y)`. Two
  numbers saying *where* that channel's activation is concentrated. For navigation this is exactly the
  right summary: it is a **bearing**, not a texture description. (The technique is from Levine et al.'s
  visuomotor policy work, used there for the same reason.)
* **per-channel max** — one number saying *whether* the feature is present at all. Without it the softmax
  centroid is meaningless when a channel is silent, since a softmax over near-uniform noise returns the
  centre of the image.
* **2x2 average pool** — four numbers per channel giving coarse quadrant structure, which the centroid
  alone cannot express (the centroid of "walls on both sides" is the middle, same as no walls at all).

With `C = 8` that is 56 spatial features plus 8 proprioceptive ones (4 heading, 3 previous action,
1 previous reward) = 64 into a `GRU(64, 20)`.

| v3 component | Parameters |
|---|---|
| `Conv2d(20 -> 8, 3x3)` | 1,448 |
| `Conv2d(8 -> 8, 3x3)` | 584 |
| `SpatialReadout` | **0** |
| `GRU(64, 20)` | 5,160 |
| `Linear(20 -> 20)` | 420 |
| `Linear(20 -> 3)` | 63 |
| **Total** | **7,675** |

A GRU is used rather than an LSTM because it has three gates instead of four — 25% cheaper for the same
width, and the recurrent core is the largest remaining block.

Everything else is inherited from v2 unchanged: one-hot categorical planes, no pooling in the conv trunk,
episode-based sequence replay, burn-in, Double-DQN targets, masked loss, truncation-aware bootstrapping.

## 4. Notebook layout

| Section | Content |
|---|---|
| 1 | Setup and imports |
| 2 | Classes — wrappers, spatial readout, tiny Q-network, sequence buffer, agent |
| 3 | Parameter initialisation, with the budget enforced as an assertion |
| 4 | Learning |
| 5 | Learning-progress plots |
| 6 | Testing / evaluation |
| 7 | Capacity sweep: does the budget actually cost you anything? |

## 1. Setup and imports

In [ ]:
# Run once if the packages are missing.
# %pip install "minigrid>=2.3" "gymnasium>=0.29" torch matplotlib numpy

In [7]:
import random
import time
from collections import deque
from dataclasses import dataclass, asdict, replace

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import gymnasium as gym
import minigrid  # noqa: F401  -- importing registers the MiniGrid environments in gymnasium
from minigrid.core.constants import OBJECT_TO_IDX, COLOR_TO_IDX, STATE_TO_IDX

print("gymnasium :", gym.__version__)
print("minigrid  :", getattr(minigrid, "__version__", "unknown"))
print("torch     :", torch.__version__)
print("cuda      :", torch.cuda.is_available())

gymnasium : 1.4.0
minigrid  : 3.1.0
torch     : 2.14.1+cpu
cuda      : False


## 2. Classes

`NavigationActions`, `CompactObsWrapper`, `PrevActionRewardWrapper`, `ObsSpec` and
`SequenceReplayBuffer` are carried over from v2 without change. The new pieces are `SpatialReadout` and
`TinyQNetwork`.

In [8]:
class NavigationActions(gym.ActionWrapper):
    # Keep only the navigation actions: 0 = turn left, 1 = turn right, 2 = move forward.
    # pickup/drop/toggle/done are no-ops in FourRooms, so dropping them removes 4 of 7 useless actions.

    def __init__(self, env, n_actions=3):
        super().__init__(env)
        self.n_actions = int(n_actions)
        self.action_space = gym.spaces.Discrete(self.n_actions)

    def action(self, action):
        return int(action)


class CompactObsWrapper(gym.ObservationWrapper):
    # Dict observation -> flat float32 vector: [147 raw image indices | 4 direction one-hot].
    # Image indices stay categorical; the network one-hot expands them on device.

    N_DIRECTIONS = 4

    def __init__(self, env):
        super().__init__(env)
        img_space = env.observation_space["image"]
        self.height, self.width, self.n_img_channels = img_space.shape   # (7, 7, 3)
        self.raw_image_size = self.height * self.width * self.n_img_channels
        self.channel_sizes = (len(OBJECT_TO_IDX), len(COLOR_TO_IDX), len(STATE_TO_IDX))
        self.obs_size = self.raw_image_size + self.N_DIRECTIONS
        self.observation_space = gym.spaces.Box(
            low=0.0, high=float(max(self.channel_sizes)), shape=(self.obs_size,), dtype=np.float32
        )

    def observation(self, obs):
        img = obs["image"].astype(np.float32).reshape(-1)     # (H, W, C) flattened H-major
        direction = np.zeros(self.N_DIRECTIONS, dtype=np.float32)
        direction[int(obs["direction"])] = 1.0               # ABSOLUTE heading -> enables dead reckoning
        return np.concatenate([img, direction]).astype(np.float32)


class PrevActionRewardWrapper(gym.Wrapper):
    # Appends [previous action one-hot | previous reward]. For a dead-reckoning agent this is essential:
    # position can only be integrated if the agent knows which move it just made.

    def __init__(self, env):
        super().__init__(env)
        self.n_actions = int(env.action_space.n)
        base_size = int(env.observation_space.shape[0])
        self.extra_size = self.n_actions + 1
        self.obs_size = base_size + self.extra_size
        self.observation_space = gym.spaces.Box(
            low=0.0, high=np.inf, shape=(self.obs_size,), dtype=np.float32
        )

    def _augment(self, obs, prev_action, prev_reward):
        extra = np.zeros(self.extra_size, dtype=np.float32)
        if prev_action is not None:
            extra[int(prev_action)] = 1.0
        extra[-1] = float(prev_reward)
        return np.concatenate([obs, extra]).astype(np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._augment(obs, None, 0.0), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        return self._augment(obs, action, reward), reward, terminated, truncated, info

In [9]:
@dataclass(frozen=True)
class ObsSpec:
    # Every dimension the network needs, passed explicitly rather than relying on
    # gymnasium attribute forwarding through the wrapper stack.
    height: int
    width: int
    n_img_channels: int
    raw_image_size: int
    channel_sizes: tuple
    n_planes: int          # sum(channel_sizes): conv input channels after one-hot expansion
    extra_size: int        # direction one-hot + prev action one-hot + prev reward
    obs_size: int
    n_actions: int

In [10]:
class SpatialReadout(nn.Module):
    # Collapses (batch, C, H, W) -> (batch, 7C) using ZERO parameters.
    #
    #   spatial softmax : 2 per channel -- expected (x, y) of the activation, i.e. a BEARING
    #   per-channel max : 1 per channel -- whether the feature is present at all
    #   2x2 avg pool    : 4 per channel -- coarse quadrant structure
    #
    # This replaces Flatten + Linear(1576, 256), which alone was 403,712 parameters in v2.
    #
    # Why all three: the softmax centroid of a silent channel is just the image centre, so the max is
    # needed to tell "feature at the middle" from "no feature". And a centroid cannot represent
    # "walls on both sides" -- that averages to the middle too -- so the pool supplies the structure.

    def __init__(self, height, width, temperature=1.0):
        super().__init__()
        self.temperature = float(temperature)
        ys, xs = torch.meshgrid(
            torch.linspace(-1.0, 1.0, height),
            torch.linspace(-1.0, 1.0, width),
            indexing="ij",
        )
        # Buffers, not plain tensors, so .to(device) moves them and they are not parameters.
        self.register_buffer("pos_x", xs.reshape(1, -1))
        self.register_buffer("pos_y", ys.reshape(1, -1))

    @staticmethod
    def output_size(n_channels):
        return 7 * int(n_channels)

    def forward(self, x):
        batch, channels, height, width = x.shape
        flat = x.reshape(batch * channels, height * width)

        weights = F.softmax(flat / self.temperature, dim=1)
        exp_x = (weights * self.pos_x).sum(dim=1)
        exp_y = (weights * self.pos_y).sum(dim=1)
        centroid = torch.stack([exp_x, exp_y], dim=1).reshape(batch, 2 * channels)

        presence = flat.max(dim=1).values.reshape(batch, channels)
        pooled = F.adaptive_avg_pool2d(x, 2).reshape(batch, 4 * channels)

        return torch.cat([centroid, presence, pooled], dim=1)

In [11]:
class TinyQNetwork(nn.Module):
    # One-hot expansion -> small conv trunk (no pooling) -> parameter-free spatial readout
    # -> GRU -> MLP head -> Q per action.
    #
    # Shapes flow as (batch, n_steps, features); a single environment step is n_steps = 1.

    def __init__(self, spec, conv_channels=8, hidden_dim=20, use_memory=True):
        super().__init__()
        self.spec = spec
        self.hidden_dim = int(hidden_dim)
        self.use_memory = bool(use_memory)

        # padding=1 keeps the map at 7x7: no row or column is discarded.
        self.conv = nn.Sequential(
            nn.Conv2d(spec.n_planes, conv_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(conv_channels, conv_channels, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.readout = SpatialReadout(spec.height, spec.width)
        feature_size = SpatialReadout.output_size(conv_channels) + spec.extra_size

        if self.use_memory:
            self.core = nn.GRU(feature_size, self.hidden_dim, batch_first=True)
        else:
            self.core = nn.Sequential(nn.Linear(feature_size, self.hidden_dim), nn.ReLU())

        self.head = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, spec.n_actions),
        )

    def initial_hidden(self, batch_size, device):
        if not self.use_memory:
            return None
        # GRU hidden state is a single tensor, not the (h, c) tuple an LSTM needs.
        return torch.zeros(1, batch_size, self.hidden_dim, device=device)

    def _expand_image(self, flat_img, batch_time):
        # (batch*time, 147) raw indices -> (batch*time, 20, 7, 7) one-hot planes.
        spec = self.spec
        idx = flat_img.reshape(batch_time, spec.height, spec.width, spec.n_img_channels).long()
        planes = []
        for channel, size in enumerate(spec.channel_sizes):
            clamped = idx[..., channel].clamp(0, size - 1)   # out-of-place: idx[...] is a view
            planes.append(F.one_hot(clamped, size))
        return torch.cat(planes, dim=-1).permute(0, 3, 1, 2).float()

    def forward(self, x, hidden=None):
        # x: (batch, n_steps, obs_size) -> q: (batch, n_steps, n_actions)
        batch, n_steps = x.shape[0], x.shape[1]
        bt = batch * n_steps
        spec = self.spec

        img = self._expand_image(x[..., :spec.raw_image_size].reshape(bt, -1), bt)
        extra = x[..., spec.raw_image_size:].reshape(bt, -1)

        spatial = self.readout(self.conv(img))
        features = torch.cat([spatial, extra], dim=1).reshape(batch, n_steps, -1)

        if self.use_memory:
            if hidden is None:
                hidden = self.initial_hidden(batch, x.device)
            core_out, hidden = self.core(features, hidden)
        else:
            core_out, hidden = self.core(features), None

        return self.head(core_out), hidden

In [12]:
class SequenceReplayBuffer:
    # Stores complete episodes and samples fixed-length sub-sequences, because a recurrent core
    # must be unrolled over contiguous time. Each episode keeps T+1 observations (so next_obs[t]
    # is obs[t+1]), T actions, T rewards and T termination flags.
    #
    # Capacity is counted in transitions; the oldest episodes are evicted first.

    def __init__(self, capacity_transitions, obs_size, device):
        self.capacity = int(capacity_transitions)
        self.obs_size = int(obs_size)
        self.device = device
        self.episodes = deque()
        self.n_transitions = 0

    def __len__(self):
        return self.n_transitions

    def push_episode(self, obs, actions, rewards, terminated):
        length = len(actions)
        if length == 0:
            return
        assert len(obs) == length + 1, "need one more observation than actions"
        self.episodes.append({
            "obs": np.asarray(obs, dtype=np.float32),
            "actions": np.asarray(actions, dtype=np.int64),
            "rewards": np.asarray(rewards, dtype=np.float32),
            "terminated": np.asarray(terminated, dtype=np.float32),
        })
        self.n_transitions += length
        while self.n_transitions > self.capacity and len(self.episodes) > 1:
            old = self.episodes.popleft()
            self.n_transitions -= len(old["actions"])

    def sample(self, batch_size, burn_in, seq_len):
        total = burn_in + seq_len
        n_obs = total + 1

        obs_seq = np.zeros((batch_size, n_obs, self.obs_size), dtype=np.float32)
        actions = np.zeros((batch_size, seq_len), dtype=np.int64)
        rewards = np.zeros((batch_size, seq_len), dtype=np.float32)
        terminated = np.zeros((batch_size, seq_len), dtype=np.float32)
        mask = np.zeros((batch_size, seq_len), dtype=np.float32)

        n_eps = len(self.episodes)
        for b in range(batch_size):
            ep = self.episodes[np.random.randint(n_eps)]
            length = len(ep["actions"])
            start = np.random.randint(0, max(1, length - total + 1))

            avail = min(n_obs, (length + 1) - start)
            obs_seq[b, :avail] = ep["obs"][start:start + avail]
            if avail < n_obs:
                obs_seq[b, avail:] = ep["obs"][-1]           # pad by repeating the final obs

            lo = start + burn_in
            hi = min(start + total, length)
            n_valid = max(0, hi - lo)
            if n_valid > 0:
                actions[b, :n_valid] = ep["actions"][lo:hi]
                rewards[b, :n_valid] = ep["rewards"][lo:hi]
                terminated[b, :n_valid] = ep["terminated"][lo:hi]
                mask[b, :n_valid] = 1.0

        to_t = lambda x, dtype: torch.as_tensor(x, dtype=dtype, device=self.device)
        return (
            to_t(obs_seq, torch.float32),
            to_t(actions, torch.int64),
            to_t(rewards, torch.float32),
            to_t(terminated, torch.float32),
            to_t(mask, torch.float32),
        )

In [13]:
class TinyDRQNAgent:
    # Recurrent Q-learning:
    #   target = r + gamma * (1 - terminated) * Q_target(s', argmax_a Q_online(s', a))   [Double DQN]
    #   loss   = mean over VALID timesteps of SmoothL1( Q_online(s, a), target )
    #
    # Sequences start from a zero hidden state and are warmed up over `burn_in` steps whose losses
    # are discarded. NOTE: the R2D2 paper (Kapturowski et al., ICLR 2019) found that STORING the
    # hidden state beats zero-start plus burn-in -- zero-start suffers worse "recurrent state
    # staleness". Zero-start is used here for simplicity; storing states is the known improvement.

    def __init__(self, spec, cfg):
        self.spec = spec
        self.cfg = cfg
        self.device = torch.device(cfg.device)
        self.n_actions = spec.n_actions

        make = lambda: TinyQNetwork(spec, cfg.conv_channels, cfg.hidden_dim, cfg.use_memory)
        self.online_net = make().to(self.device)
        self.target_net = make().to(self.device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = torch.optim.Adam(self.online_net.parameters(), lr=cfg.lr)
        self.buffer = SequenceReplayBuffer(cfg.buffer_size, spec.obs_size, self.device)
        self.train_steps = 0

    def epsilon(self, step):
        frac = min(1.0, step / max(1, self.cfg.eps_decay_steps))
        return self.cfg.eps_start + frac * (self.cfg.eps_end - self.cfg.eps_start)

    def initial_hidden(self, batch_size=1):
        return self.online_net.initial_hidden(batch_size, self.device)

    @torch.no_grad()
    def act(self, obs, hidden, epsilon=0.0):
        # The forward pass runs even for a random action: the GRU must still consume this
        # observation or the hidden state would skip a timestep.
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=self.device).view(1, 1, -1)
        q_values, hidden = self.online_net(obs_t, hidden)
        if random.random() < epsilon:
            action = random.randrange(self.n_actions)
        else:
            action = int(q_values[0, -1].argmax().item())
        return action, hidden

    def update(self):
        cfg = self.cfg
        if len(self.buffer) < max(cfg.learning_starts, cfg.burn_in + cfg.seq_len + 1):
            return None
        if len(self.buffer.episodes) == 0:
            return None

        obs_seq, actions, rewards, terminated, mask = self.buffer.sample(
            cfg.batch_size, cfg.burn_in, cfg.seq_len
        )
        batch = obs_seq.shape[0]

        # --- burn-in: warm the hidden state without contributing gradients -------------------
        if cfg.burn_in > 0:
            with torch.no_grad():
                _, hidden_online = self.online_net(
                    obs_seq[:, :cfg.burn_in], self.online_net.initial_hidden(batch, self.device)
                )
                _, hidden_target = self.target_net(
                    obs_seq[:, :cfg.burn_in], self.target_net.initial_hidden(batch, self.device)
                )
        else:
            hidden_online = self.online_net.initial_hidden(batch, self.device)
            hidden_target = self.target_net.initial_hidden(batch, self.device)

        # --- scored window: seq_len + 1 observations give Q(s_t) and Q(s_{t+1}) --------------
        q_all, _ = self.online_net(obs_seq[:, cfg.burn_in:], hidden_online)
        q_taken = q_all[:, :cfg.seq_len].gather(2, actions.unsqueeze(2)).squeeze(2)

        with torch.no_grad():
            target_all, _ = self.target_net(obs_seq[:, cfg.burn_in:], hidden_target)
            if cfg.double_dqn:
                next_actions = q_all[:, 1:].detach().argmax(dim=2, keepdim=True)
                next_q = target_all[:, 1:].gather(2, next_actions).squeeze(2)
            else:
                next_q = target_all[:, 1:].max(dim=2).values
            targets = rewards + cfg.gamma * (1.0 - terminated) * next_q

        elementwise = F.smooth_l1_loss(q_taken, targets, reduction="none")
        loss = (elementwise * mask).sum() / mask.sum().clamp(min=1.0)

        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), cfg.max_grad_norm)
        self.optimizer.step()

        self.train_steps += 1
        if self.train_steps % cfg.target_update_every == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())

        return float(loss.item())

    def save(self, path):
        torch.save(self.online_net.state_dict(), path)

    def load(self, path):
        state = torch.load(path, map_location=self.device)
        self.online_net.load_state_dict(state)
        self.target_net.load_state_dict(state)

## 3. Parameter initialisation

The parameter budget is enforced as an **assertion**, not a comment — if you widen anything past 10,000
the cell fails rather than quietly spending your budget.

`gamma = 0.997` and `seq_len = 32` follow DI-engine's published R2D2 MiniGrid config rather than v2's
values: reward arrives tens of steps after the decisions that earn it, so a longer horizon and a longer
unroll both help.

In [14]:
QUICK_RUN = False        # True -> a short run that only checks the pipeline works
PARAM_BUDGET = 10_000


@dataclass
class Config:
    # environment -- fixed start and goal, but the maze layout still varies (4,096 variants)
    env_id: str = "MiniGrid-FourRooms-v0"
    agent_pos: tuple = (1, 1)
    goal_pos: tuple = (17, 17)
    seed: int = 0
    n_actions: int = 3

    # training budget
    num_episodes: int = 6000

    # Q-learning
    gamma: float = 0.997             # DI-engine R2D2 value; reward is far from the causal action
    lr: float = 5e-4                 # small nets tolerate, and need, a slightly larger step
    batch_size: int = 32             # sequences per gradient step
    buffer_size: int = 60_000        # capacity in transitions
    learning_starts: int = 3_000
    train_every: int = 4
    target_update_every: int = 1_000
    double_dqn: bool = True
    max_grad_norm: float = 10.0

    # recurrence
    use_memory: bool = True
    seq_len: int = 32                # longer than v2's 16: dead reckoning needs a long horizon
    burn_in: int = 4

    # exploration
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_decay_steps: int = 150_000

    # network -- these two set the parameter count
    conv_channels: int = 8
    hidden_dim: int = 20

    # misc
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log_every: int = 100


cfg = Config()
if QUICK_RUN:
    cfg = replace(cfg, num_episodes=400, eps_decay_steps=12_000,
                  learning_starts=1_000, log_every=25)

for key, value in asdict(cfg).items():
    print(f"{key:22s} = {value}")

env_id                 = MiniGrid-FourRooms-v0
agent_pos              = (1, 1)
goal_pos               = (17, 17)
seed                   = 0
n_actions              = 3
num_episodes           = 6000
gamma                  = 0.997
lr                     = 0.0005
batch_size             = 32
buffer_size            = 60000
learning_starts        = 3000
train_every            = 4
target_update_every    = 1000
double_dqn             = True
max_grad_norm          = 10.0
use_memory             = True
seq_len                = 32
burn_in                = 4
eps_start              = 1.0
eps_end                = 0.05
eps_decay_steps        = 150000
conv_channels          = 8
hidden_dim             = 20
device                 = cpu
log_every              = 100


In [15]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_env(cfg, render_mode=None):
    # agent_pos and goal_pos are fixed. The four doorway positions and the start heading are NOT --
    # MiniGrid re-randomises them in _gen_grid on every reset regardless of these kwargs.
    env = gym.make(
        cfg.env_id,
        agent_pos=tuple(cfg.agent_pos),
        goal_pos=tuple(cfg.goal_pos),
        render_mode=render_mode,
    )
    env = NavigationActions(env, cfg.n_actions)
    inner = CompactObsWrapper(env)
    env = PrevActionRewardWrapper(inner)

    spec = ObsSpec(
        height=inner.height,
        width=inner.width,
        n_img_channels=inner.n_img_channels,
        raw_image_size=inner.raw_image_size,
        channel_sizes=inner.channel_sizes,
        n_planes=sum(inner.channel_sizes),
        extra_size=inner.N_DIRECTIONS + env.extra_size,
        obs_size=env.obs_size,
        n_actions=int(env.action_space.n),
    )
    return env, spec


set_seed(cfg.seed)
env, spec = make_env(cfg)

obs, info = env.reset(seed=cfg.seed)
env.action_space.seed(cfg.seed)

print("observation vector size :", obs.shape)
print("grid size               :", env.unwrapped.width, "x", env.unwrapped.height)
print("max steps per episode   :", env.unwrapped.max_steps)
print("agent start (fixed)     :", tuple(env.unwrapped.agent_pos))
print("agent heading (RANDOM)  :", env.unwrapped.agent_dir)

observation vector size : (155,)
grid size               : 19 x 19
max steps per episode   : 100
agent start (fixed)     : (1, 1)
agent heading (RANDOM)  : 1


In [16]:
# Confirm empirically that the layout really does vary while start and goal do not.
def grid_fingerprint(environment):
    # Hash the wall pattern so differing layouts are visible.
    walls = tuple(
        1 if (obj is not None and obj.type == "wall") else 0
        for obj in environment.unwrapped.grid.grid
    )
    return hash(walls)


probe_env, _ = make_env(cfg)
layouts, starts, headings = set(), set(), set()
for k in range(200):
    probe_env.reset(seed=1000 + k)
    layouts.add(grid_fingerprint(probe_env))
    starts.add(tuple(probe_env.unwrapped.agent_pos))
    headings.add(int(probe_env.unwrapped.agent_dir))
probe_env.close()

print(f"distinct wall layouts over 200 resets : {len(layouts)}  (expect many -- up to 4,096 exist)")
print(f"distinct agent start cells            : {len(starts)} -> {starts}")
print(f"distinct start headings               : {len(headings)} -> {sorted(headings)}")
assert len(starts) == 1, "agent_pos kwarg should pin the start cell"

distinct wall layouts over 200 resets : 197  (expect many -- up to 4,096 exist)
distinct agent start cells            : 1 -> {(1, 1)}
distinct start headings               : 4 -> [0, 1, 2, 3]


In [17]:
agent = TinyDRQNAgent(spec, cfg)

n_params = sum(p.numel() for p in agent.online_net.parameters() if p.requires_grad)
print(agent.online_net)
print()
for name, module in [("conv trunk", agent.online_net.conv),
                     ("spatial readout", agent.online_net.readout),
                     ("GRU core", agent.online_net.core),
                     ("MLP head", agent.online_net.head)]:
    count = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"  {name:<18}{count:>8,}")
print(f"  {'TOTAL':<18}{n_params:>8,}   budget {PARAM_BUDGET:,}")

assert n_params < PARAM_BUDGET, f"over budget: {n_params:,} >= {PARAM_BUDGET:,}"
assert sum(p.numel() for p in agent.online_net.readout.parameters()) == 0, \
    "the spatial readout must be parameter-free"

# Shape sanity check before committing to a long run.
with torch.no_grad():
    dummy = torch.as_tensor(obs, dtype=torch.float32, device=agent.device).view(1, 1, -1)
    q, h = agent.online_net(dummy, agent.initial_hidden(1))
    print("\nsingle-step Q shape :", tuple(q.shape), "(expected (1, 1, 3))")
    if cfg.use_memory:
        print("GRU hidden shape    :", tuple(h.shape))

TinyQNetwork(
  (conv): Sequential(
    (0): Conv2d(20, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (readout): SpatialReadout()
  (core): GRU(64, 20, batch_first=True)
  (head): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=3, bias=True)
  )
)

  conv trunk           2,032
  spatial readout          0
  GRU core             5,160
  MLP head               483
  TOTAL                7,675   budget 10,000

single-step Q shape : (1, 1, 3) (expected (1, 1, 3))
GRU hidden shape    : (1, 1, 20)


## 4. Learning

Identical to v2's loop. The two details that are easy to get wrong:

* the hidden state is reset at every episode boundary and carried forward within an episode, and is
  updated even when the chosen action is random;
* only `terminated` cuts the bootstrap, never `truncated` — the 100-step limit is not a real terminal
  state, the agent simply ran out of time to collect the remaining value.

In [18]:
def train(agent, env, cfg):
    history = {k: [] for k in
               ("episode_return", "episode_length", "success", "epsilon", "loss", "global_step")}
    global_step = 0
    start_time = time.time()

    for episode in range(1, cfg.num_episodes + 1):
        obs, _ = env.reset()
        hidden = agent.initial_hidden(1)

        obs_list = [obs]
        action_list, reward_list, terminated_list = [], [], []
        recent_losses = []

        ep_return, ep_len = 0.0, 0
        eps = agent.epsilon(global_step)          # captured here so a zero-step episode is safe
        episode_terminated = False
        done = False

        while not done:
            eps = agent.epsilon(global_step)
            action, hidden = agent.act(obs, hidden, eps)

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = bool(terminated or truncated)
            episode_terminated = bool(terminated)

            obs_list.append(next_obs)
            action_list.append(action)
            reward_list.append(float(reward))
            terminated_list.append(float(terminated))    # NOT `done`

            obs = next_obs
            ep_return += float(reward)
            ep_len += 1
            global_step += 1

            if global_step % cfg.train_every == 0:
                loss = agent.update()
                if loss is not None:
                    recent_losses.append(loss)

        agent.buffer.push_episode(obs_list, action_list, reward_list, terminated_list)

        history["episode_return"].append(ep_return)
        history["episode_length"].append(ep_len)
        history["success"].append(1.0 if episode_terminated else 0.0)
        history["epsilon"].append(eps)
        history["loss"].append(float(np.mean(recent_losses)) if recent_losses else np.nan)
        history["global_step"].append(global_step)

        if episode % cfg.log_every == 0:
            w = min(cfg.log_every, len(history["episode_return"]))
            recent_loss = [x for x in history["loss"][-w:] if not np.isnan(x)]
            loss_str = f"{np.mean(recent_loss):.4f}" if recent_loss else "   n/a"
            print(
                f"ep {episode:5d} | step {global_step:7d} | eps {eps:.3f} | "
                f"return {np.mean(history['episode_return'][-w:]):.3f} | "
                f"success {np.mean(history['success'][-w:]):.2f} | "
                f"len {np.mean(history['episode_length'][-w:]):5.1f} | "
                f"loss {loss_str} | {time.time() - start_time:6.0f}s"
            )

    return history

In [19]:
history = train(agent, env, cfg)
agent.save("fourrooms_tiny_drqn.pt")
print(f"training finished; {n_params:,} parameters saved to fourrooms_tiny_drqn.pt")

ep   100 | step   10000 | eps 0.937 | return 0.000 | success 0.00 | len 100.0 | loss 0.0071 |     33s
ep   200 | step   20000 | eps 0.873 | return 0.000 | success 0.00 | len 100.0 | loss 0.0001 |     80s
ep   300 | step   30000 | eps 0.810 | return 0.000 | success 0.00 | len 100.0 | loss 0.0000 |    128s
ep   400 | step   40000 | eps 0.747 | return 0.000 | success 0.00 | len 100.0 | loss 0.0000 |    176s
ep   500 | step   50000 | eps 0.683 | return 0.000 | success 0.00 | len 100.0 | loss 0.0000 |    224s
ep   600 | step   60000 | eps 0.620 | return 0.000 | success 0.00 | len 100.0 | loss 0.0000 |    272s
ep   700 | step   70000 | eps 0.557 | return 0.000 | success 0.00 | len 100.0 | loss 0.0000 |    319s
ep   800 | step   80000 | eps 0.493 | return 0.000 | success 0.00 | len 100.0 | loss 0.0000 |    365s
ep   900 | step   90000 | eps 0.430 | return 0.000 | success 0.00 | len 100.0 | loss 0.0000 |    414s
ep  1000 | step  100000 | eps 0.367 | return 0.000 | success 0.00 | len 100.0 | lo

KeyboardInterrupt: 

## 5. Learning progress

In [ ]:
def moving_average(values, window):
    # NaN-aware: averages over the valid entries in each window, so "no gradient update yet"
    # is not silently reported as a loss of 0.0.
    v = np.asarray(values, dtype=float)
    if len(v) < window:
        return np.array([]), np.array([])
    valid = ~np.isnan(v)
    filled = np.where(valid, v, 0.0)
    kernel = np.ones(window)
    total = np.convolve(filled, kernel, mode="valid")
    count = np.convolve(valid.astype(float), kernel, mode="valid")
    with np.errstate(invalid="ignore", divide="ignore"):
        smoothed = np.where(count > 0, total / np.maximum(count, 1.0), np.nan)
    return np.arange(window, len(v) + 1), smoothed


def plot_history(history, window=100, title=None):
    if title is None:
        title = f"Tiny recurrent DQN ({n_params:,} parameters) on fixed-goal FourRooms"
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    episodes = np.arange(1, len(history["episode_return"]) + 1)
    window = max(1, min(window, len(episodes)))

    ax = axes[0, 0]
    ax.plot(episodes, history["episode_return"], color="tab:blue", alpha=0.20, lw=0.8, label="per episode")
    x, y = moving_average(history["episode_return"], window)
    if len(x):
        ax.plot(x, y, color="tab:blue", lw=2, label=f"moving avg ({window})")
    ax.set_xlabel("episode"); ax.set_ylabel("return")
    ax.set_title("Episode return"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[0, 1]
    x, y = moving_average(history["success"], window)
    if len(x):
        ax.plot(x, y, color="tab:green", lw=2)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel("episode"); ax.set_ylabel("goal reached (fraction)")
    ax.set_title(f"Success rate (moving avg, window={window})"); ax.grid(alpha=0.3)

    ax = axes[1, 0]
    ax.plot(episodes, history["episode_length"], color="tab:orange", alpha=0.20, lw=0.8)
    x, y = moving_average(history["episode_length"], window)
    if len(x):
        ax.plot(x, y, color="tab:orange", lw=2)
    ax.set_xlabel("episode"); ax.set_ylabel("steps")
    ax.set_title("Episode length (lower = goal found faster)"); ax.grid(alpha=0.3)

    ax = axes[1, 1]
    ax.plot(episodes, history["epsilon"], color="tab:red", lw=2)
    ax.set_xlabel("episode"); ax.set_ylabel("epsilon", color="tab:red")
    ax.tick_params(axis="y", labelcolor="tab:red")
    ax.set_ylim(0, 1.05); ax.grid(alpha=0.3)
    ax2 = ax.twinx()
    x, y = moving_average(history["loss"], window)
    if len(x):
        ax2.plot(x, y, color="tab:purple", lw=1.5)
    ax2.set_ylabel("smoothed TD loss", color="tab:purple")
    ax2.tick_params(axis="y", labelcolor="tab:purple")
    ax.set_title("Exploration and TD loss")

    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()
    return fig


fig = plot_history(history)
fig.savefig("fourrooms_tiny_learning_curve.png", dpi=150, bbox_inches="tight")

## 6. Testing

Greedy evaluation against a random baseline. The baseline samples from the **evaluation** env's own action
space, not the training env's.

For context, published numbers on the *harder* default variant (random goal): PPO with 5M steps reaches
0.52 +/- 0.29 and R2D2 with 10M steps exceeds 0.6. This fixed-goal variant should beat both — that is the
point of fixing the goal — so treat those as a floor rather than a target.

In [ ]:
def evaluate(cfg, policy, n_episodes=200, seed=10_000, needs_hidden=False, agent=None):
    eval_env, _ = make_env(cfg)
    obs, _ = eval_env.reset(seed=seed)
    eval_env.action_space.seed(seed)

    returns, lengths, successes, success_lengths = [], [], [], []

    for episode in range(n_episodes):
        if episode > 0:
            obs, _ = eval_env.reset()
        hidden = agent.initial_hidden(1) if needs_hidden else None
        ep_return, ep_len = 0.0, 0
        episode_terminated = False
        done = False

        while not done:
            action, hidden = policy(obs, hidden) if needs_hidden else (policy(eval_env), None)
            obs, reward, terminated, truncated, _ = eval_env.step(action)
            done = bool(terminated or truncated)
            episode_terminated = bool(terminated)
            ep_return += float(reward)
            ep_len += 1

        returns.append(ep_return)
        lengths.append(ep_len)
        successes.append(1.0 if episode_terminated else 0.0)
        if episode_terminated:
            success_lengths.append(ep_len)

    eval_env.close()
    return {
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "success_rate": float(np.mean(successes)),
        "mean_length": float(np.mean(lengths)),
        "mean_length_on_success": float(np.mean(success_lengths)) if success_lengths else float("nan"),
        "returns": returns,
    }


agent.online_net.eval()
greedy_stats = evaluate(cfg, lambda o, h: agent.act(o, h, epsilon=0.0),
                        n_episodes=200, seed=10_000, needs_hidden=True, agent=agent)
random_stats = evaluate(cfg, lambda e: e.action_space.sample(),
                        n_episodes=200, seed=10_000, needs_hidden=False)
agent.online_net.train()

header = f"{'metric':<26}{'tiny DRQN':>14}{'random':>12}"
print(header); print("-" * len(header))
for key, label in [("mean_return", "mean return"), ("std_return", "std return"),
                   ("success_rate", "success rate"), ("mean_length", "mean episode length"),
                   ("mean_length_on_success", "mean length (successes)")]:
    print(f"{label:<26}{greedy_stats[key]:>14.3f}{random_stats[key]:>12.3f}")

print(f"\nreturn per 1,000 parameters: {1000 * greedy_stats['mean_return'] / n_params:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 1, 21)
ax.hist(random_stats["returns"], bins=bins, alpha=0.6, label="random policy", color="tab:gray")
ax.hist(greedy_stats["returns"], bins=bins, alpha=0.6,
        label=f"tiny DRQN ({n_params:,} params)", color="tab:blue")
ax.set_xlabel("episode return"); ax.set_ylabel("number of episodes")
ax.set_title("Evaluation over 200 episodes"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 7. Capacity sweep — does the budget actually cost anything?

The claim behind this notebook is that ~7.7k parameters is *enough*, not merely *possible*. The way to
test that is a sweep: train the same agent at several widths and see where performance saturates. If the
curve is flat from 7k to 60k, the budget costs nothing. If it climbs steeply, it does.

Each configuration is a full training run, so this cell is expensive — budget accordingly and reduce
`num_episodes` before enabling it.

In [ ]:
RUN_SWEEP = False

SWEEP = [
    dict(conv_channels=4,  hidden_dim=12),   #  2,867 params
    dict(conv_channels=8,  hidden_dim=20),   #  7,675 params  (the shipped default)
    dict(conv_channels=16, hidden_dim=48),   # 32,195 params
    dict(conv_channels=8,  hidden_dim=20, use_memory=False),   # memoryless control
]

if RUN_SWEEP:
    results = []
    for i, overrides in enumerate(SWEEP):
        sweep_cfg = replace(cfg, num_episodes=max(500, cfg.num_episodes // 4), **overrides)
        set_seed(cfg.seed + 100 + i)
        sweep_env, sweep_spec = make_env(sweep_cfg)
        sweep_env.reset(seed=cfg.seed + 100 + i)

        sweep_agent = TinyDRQNAgent(sweep_spec, sweep_cfg)
        count = sum(p.numel() for p in sweep_agent.online_net.parameters() if p.requires_grad)
        label = (f"C={overrides.get('conv_channels')} h={overrides.get('hidden_dim')}"
                 f"{'' if overrides.get('use_memory', True) else ' (no memory)'}")
        print(f"\n=== {label}: {count:,} parameters ===")

        hist = train(sweep_agent, sweep_env, sweep_cfg)
        sweep_agent.online_net.eval()
        stats = evaluate(sweep_cfg, lambda o, h: sweep_agent.act(o, h, epsilon=0.0),
                         n_episodes=100, seed=10_000, needs_hidden=True, agent=sweep_agent)
        results.append((label, count, stats["mean_return"], stats["success_rate"], hist))
        sweep_env.close()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    for label, count, _, _, hist in results:
        x, y = moving_average(hist["success"], 50)
        if len(x):
            ax1.plot(x, y, lw=2, label=f"{label} ({count:,})")
    ax1.set_xlabel("episode"); ax1.set_ylabel("success rate (moving avg, 50)")
    ax1.set_title("Learning curves by capacity"); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    counts = [r[1] for r in results]
    ax2.plot(counts, [r[2] for r in results], "o-", lw=2, color="tab:blue")
    ax2.set_xscale("log")
    ax2.set_xlabel("trainable parameters (log scale)"); ax2.set_ylabel("greedy mean return")
    ax2.set_title("Does more capacity buy return?"); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"\n{'configuration':<32}{'params':>10}{'return':>10}{'success':>10}")
    for label, count, ret, succ, _ in results:
        print(f"{label:<32}{count:>10,}{ret:>10.3f}{succ:>10.3f}")
else:
    print("sweep skipped -- set RUN_SWEEP = True to run it")

## Notes, and what is not yet verified

**What this notebook does and does not claim.** The parameter count is exact and asserted at runtime. The
claim that 7,675 parameters *suffice* to reach a good return on this variant is a hypothesis, and section 7
is the experiment that tests it. I have not found any published sub-10k result on any MiniGrid task, so
there is no external number to compare against — you would be measuring something new.

**Expect optimisation to be the binding constraint, not capacity.** Overparameterised networks have
better-conditioned loss landscapes and are more forgiving of learning-rate choices. A 7.7k-parameter
network can *represent* a good policy; whether Adam *finds* it from sparse rewards is a separate question.
If the run stalls, try the `C=16, h=48` sweep point (32,195 parameters) before concluding the task is
hard — and if the big one also stalls, the problem is the hyper-parameters, not the size.

**The Q-function is the capacity risk, not the policy.** The network must value every action in every
state it visits, including the incompetent states of early training, while the target keeps moving. That
target function is far richer than the final policy. If capacity binds anywhere, it binds here.

**Known weakness, inherited deliberately.** Sequences start from a zero hidden state plus burn-in. The
R2D2 paper found that *storing* the hidden state in the buffer beats this, and that zero-start suffers
noticeably worse recurrent-state staleness. Storing states is the single most defensible upgrade to this
implementation.

**Improvements ranked by expected value**, following DI-engine's published R2D2 MiniGrid config, which
uses the first two as standard rather than optional:

1. **n-step returns** (`n = 5`) — with reward tens of steps from the causal action, one-step bootstrapping
   propagates credit far too slowly. Cheap to add.
2. **Prioritised sequence replay** (`alpha = 0.6`, `beta = 0.4`) — the handful of sequences containing the
   goal transition deserve far more replay than the rest.
3. **Stored hidden states** — see above.
4. **Soft target updates** (`theta = 0.001`) instead of hard syncs every 1,000 gradient steps.
5. **Vectorised environments** — not an algorithmic change, but the acting loop is the wall-clock
   bottleneck (every env step needs a forward pass and cannot be batched across time). Sixteen parallel
   envs cut the acting cost by roughly an order of magnitude, and a GPU alone does not.

**Runtime.** Every environment step needs a GRU forward even for random actions, so plan for hours on CPU.
Set `QUICK_RUN = True` first: if it completes, the plumbing is sound.